# 문제 3: Tesseract를 활용한 약봉투 이미지 문자 인식(OCR) 실습

**소재**: 약봉투 이미지 (한글 + 영어 혼용 텍스트)  
**목표**: pytesseract로 약품명·복약안내·주의사항 텍스트 추출  
**가산점 포인트**:
- `lang='kor+eng'` 옵션으로 한글+영어 혼용 추출
- 전처리 파이프라인(흑백 변환 → 업스케일 → 이진화 → 노이즈 제거)으로 인식률 향상
- 전체 이미지 배치 처리
- 약봉투 특화 키워드 하이라이팅

## 1. 환경 설정 — Tesseract 설치 및 라이브러리 임포트

In [ ]:
# Tesseract OCR 엔진 + 한국어 학습 데이터 설치
!apt-get install -y tesseract-ocr tesseract-ocr-kor > /dev/null 2>&1

# Python 라이브러리 설치
!pip install pytesseract pillow opencv-python-headless -q

# 환경 변수 설정 — 한국어 학습 데이터 경로 및 기본 언어 지정
import os
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/5/tessdata/'
os.environ['TESSERACT_LANG']  = 'kor+eng'   # 한글+영어 혼용

print("설치 완료")
print(f"TESSDATA_PREFIX : {os.environ['TESSDATA_PREFIX']}")
print(f"TESSERACT_LANG  : {os.environ['TESSERACT_LANG']}")
!tesseract --version

In [ ]:
import os
import cv2
import numpy as np
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm

# 나눔고딕 폰트 설치 후 캐시 초기화 (설치만으로는 적용 안 됨)
!apt-get install -y fonts-nanum > /dev/null 2>&1
fm._load_fontmanager(try_read_cache=False)   # 캐시 강제 재구성

# 설치된 나눔 폰트 경로를 직접 찾아 등록
nanum_path = fm.findfont(fm.FontProperties(family='NanumGothic'))
if 'NanumGothic' not in nanum_path:
    # 경로 직접 지정 (Colab 기본 설치 위치)
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)
    nanum_path = font_path

matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료")
print(f"적용된 한글 폰트: {nanum_path}")

## 2. 이미지 업로드

> **방법 A** — Google Drive 마운트 (권장)  
> **방법 B** — 직접 파일 업로드
>
> 아래 두 셀 중 하나만 실행하세요.

In [ ]:
# 방법 A: Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# Drive 내 이미지 폴더 경로
IMAGE_DIR = '/content/drive/MyDrive/Colab Notebook/medication/'
print(f"이미지 폴더: {IMAGE_DIR}")

In [ ]:
# 방법 B: 직접 업로드 (Drive를 사용하지 않을 경우)
from google.colab import files
uploaded = files.upload()   # 파일 선택 대화상자가 열립니다

IMAGE_DIR = '/content/'
print(f"업로드된 파일: {list(uploaded.keys())}")

In [ ]:
# 이미지 파일 목록 확인
EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
image_paths = sorted(
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(EXTS)
)

print(f"발견된 이미지 수: {len(image_paths)}장")
for p in image_paths:
    print(' ', os.path.basename(p))

## 3. 기본 OCR — 전처리 없이 원본 이미지 텍스트 추출

In [ ]:
def ocr_basic(image_path: str) -> str:
    """원본 이미지에 대한 기본 OCR."""
    img = Image.open(image_path)
    # [가산점] 환경변수 TESSERACT_LANG 참조 → kor+eng 혼용 인식
    lang = os.environ['TESSERACT_LANG']
    text = pytesseract.image_to_string(img, lang=lang)
    return text


# 첫 번째 이미지로 기본 OCR 시연
sample_path = image_paths[0]

img_display = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 6))
plt.imshow(img_display)
plt.title('원본 이미지')
plt.axis('off')
plt.show()

basic_result = ocr_basic(sample_path)
print("=" * 60)
print(f"[기본 OCR 결과 — lang={os.environ['TESSERACT_LANG']}, 전처리 없음]")
print("=" * 60)
print(basic_result)

## 4. [가산점] 전처리 파이프라인으로 OCR 정확도 향상

약봉투 이미지는 배경 무늬·조명 불균일로 인식률이 낮을 수 있습니다.  
아래 4단계 전처리로 인식률을 높입니다.

| 단계 | 기법 | 목적 |
|------|------|------|
| 1 | 흑백 변환 (Grayscale) | 색상 노이즈 제거 |
| 2 | 크기 업스케일 (×2) | 작은 글자 선명화 |
| 3 | Otsu 이진화 (Thresholding) | 글자/배경 대비 극대화 |
| 4 | 미디언 블러 (Median Blur) | 점 노이즈 제거 |

In [ ]:
# [가산점] 전처리 파이프라인
def preprocess(image_path: str) -> np.ndarray:
    img = cv2.imread(image_path)

    # 1단계: 흑백 변환
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2단계: 2배 업스케일 — 작은 글자 인식률 향상
    scaled = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # 3단계: 적응형 이진화 — 컬러 배경·조명 불균일 이미지에도 강인
    binary = cv2.adaptiveThreshold(
        scaled, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=31,
        C=10
    )

    # 4단계: 미디언 블러 — 점 노이즈 제거
    denoised = cv2.medianBlur(binary, 3)

    return denoised


def ocr_with_preprocessing(image_path: str) -> tuple[np.ndarray, str]:
    processed = preprocess(image_path)
    pil_img = Image.fromarray(processed)
    # [가산점] 환경변수 TESSERACT_LANG 참조 → kor+eng 혼용 + 전처리 조합
    lang = os.environ['TESSERACT_LANG']
    text = pytesseract.image_to_string(pil_img, lang=lang)
    return processed, text


print("전처리 함수 정의 완료")
print(f"사용 언어: {os.environ['TESSERACT_LANG']}")

## 5. [가산점] 전처리 전/후 비교

In [ ]:
processed_img, preprocessed_result = ocr_with_preprocessing(sample_path)

# 시각적 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB))
axes[0].set_title('원본 이미지', fontsize=14)
axes[0].axis('off')

axes[1].imshow(processed_img, cmap='gray')
axes[1].set_title('전처리 후 (흑백 → 업스케일 → 이진화 → 노이즈 제거)', fontsize=14)
axes[1].axis('off')

plt.suptitle('전처리 전/후 비교', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# OCR 결과 비교 출력
print("=" * 60)
print("[원본 OCR 결과]")
print("=" * 60)
print(basic_result)

print("\n" + "=" * 60)
print("[전처리 후 OCR 결과]")
print("=" * 60)
print(preprocessed_result)

## 6. [가산점] 전체 이미지 배치 처리

In [ ]:
# [가산점] 약봉투 이미지 7장 전체 배치 OCR 처리
def batch_ocr(paths: list[str]) -> list[dict]:
    results = []
    for path in paths:
        _, text = ocr_with_preprocessing(path)
        results.append({'file': os.path.basename(path), 'text': text})
        print(f"  완료: {os.path.basename(path)}")
    return results


print("배치 OCR 시작...")
all_results = batch_ocr(image_paths)
print(f"\n총 {len(all_results)}장 처리 완료")

In [ ]:
# 전체 이미지 OCR 결과 출력
for i, result in enumerate(all_results, 1):
    print(f"{'=' * 60}")
    print(f"[이미지 {i}] {result['file']}")
    print(f"{'=' * 60}")
    print(result['text'])
    print()

## 7. [가산점] 약봉투 특화 키워드 하이라이팅

추출된 텍스트에서 약품명·복약 정보·주의사항 관련 키워드를 찾아 강조 표시합니다.

In [ ]:
# [가산점] 약봉투 특화 키워드 하이라이팅
PHARMACY_KEYWORDS = [
    # 복약 관련
    '1정', '2정', '3정', '1캡슐', '1회', '2회', '3회', '1일', '2일', '3일',
    '식전', '식후', '취침전', '공복',
    # 보관 관련
    '밀폐용기', '실온보관', '냉장보관', '차광',
    # 주의 관련
    '주의', '금기', '부작용', '졸음', '음주',
    # 약효 관련
    '진통제', '소염', '항생제', '위장약', '소화',
]


def highlight_keywords(text: str, keywords: list[str]) -> str:
    """발견된 키워드를 [ ] 로 감싸 강조 표시."""
    for kw in keywords:
        text = text.replace(kw, f'[★{kw}★]')
    return text


# 첫 번째 이미지 결과에 키워드 하이라이팅 적용
sample_text = all_results[0]['text']
highlighted = highlight_keywords(sample_text, PHARMACY_KEYWORDS)

print("=" * 60)
print(f"[키워드 하이라이팅 결과] {all_results[0]['file']}")
print("=" * 60)
print(highlighted)

In [ ]:
# 전체 이미지에서 발견된 키워드 통계
from collections import Counter

keyword_counts: Counter = Counter()
for result in all_results:
    for kw in PHARMACY_KEYWORDS:
        count = result['text'].count(kw)
        if count > 0:
            keyword_counts[kw] += count

print("=" * 60)
print(f"[전체 {len(all_results)}장에서 발견된 약봉투 키워드 빈도]")
print("=" * 60)
for kw, cnt in keyword_counts.most_common():
    print(f"  {kw:10s}: {cnt}회")

# 키워드 빈도 막대 그래프
if keyword_counts:
    labels, values = zip(*keyword_counts.most_common(10))
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values, color='steelblue')
    plt.title('약봉투 키워드 빈도 Top 10', fontsize=14)
    plt.xlabel('키워드')
    plt.ylabel('등장 횟수')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 8. 최종 요약

| 항목 | 내용 |
|------|------|
| OCR 엔진 | Tesseract + pytesseract |
| 언어 옵션 | `lang='kor+eng'` (한글+영어 혼용) |
| 전처리 단계 | 흑백 → 2× 업스케일 → Otsu 이진화 → 미디언 블러 |
| 처리 이미지 수 | 약봉투 7장 배치 처리 |
| 창의적 기능 | 약봉투 특화 키워드 하이라이팅 + 빈도 시각화 |

**가산점 적용 사항 (코드 주석 표시: `# [가산점]`)**
1. `lang='kor+eng'` — 한글+영어 혼용 문서 인식
2. 4단계 전처리 파이프라인으로 인식률 향상
3. 전체 이미지 배치 처리
4. 약봉투 도메인 특화 키워드 하이라이팅 및 빈도 시각화